# Lab — Support Vector Machines from scratch
**Machine Learning SUP'COM workshop**

> Logistic Regression and SVM solve the **same classification problem**, but they care about **different things**.

- Logistic Regression asks: *what is the probability of class 1?*
- SVM asks: *where is the boundary that leaves the **widest safe zone** between the classes?*

How this lab works (same rules as the logistic regression lab):

- Fill in every `# TODO` and replace each `raise NotImplementedError`.
- Run the cells **in order**.
- Compare your outputs with each **Check yourself** cell before moving on.
- Cells marked `# ---- PROVIDED` are complete: just run and read them.
- Write short answers where you see *Your answer:* ...

In [ ]:
# ---- PROVIDED
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True)

X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=2.5,
    random_state=42
)

x1_line = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k", label="class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red",  edgecolor="k", label="class 1")
for slope, intercept in [(0.0, 0.0), (0.6, -2.2), (-0.3, 0.6)]:
    plt.plot(x1_line, slope * x1_line + intercept, "--")
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.title("Several lines separate the data. Which is best?")
plt.legend()
plt.show()

Same dataset as the logistic regression lab. All three dashed lines separate the classes perfectly.

> **Which one would you trust most on new data, and why?**

*Your answer:* ...

## Part A. Score, labels and margin

### Task 1. Labels in $\{-1, +1\}$

Like logistic regression, a linear SVM computes a score

$$
f(x) = w^T x + b
$$

and $f(x) = 0$ is the decision boundary. The prediction is the **sign** of $f(x)$: no sigmoid, no probability.

SVM writes the classes as $y \in \{-1, +1\}$ because then **one expression**, $y_i f(x_i)$, is positive when the point is correctly classified and negative when it is not.

In [ ]:
# TODO: y_svm = -1 where y == 0, +1 where y == 1 (hint: np.where), in one line
y_svm = None

print("original labels:", y[:8])
print("SVM labels:     ", y_svm[:8])

**Check yourself:** `[-1 -1 1 -1 -1 -1 -1 1]`.

> Why would the trick $y_i f(x_i)$ **not** work with labels in $\{0, 1\}$?

*Your answer:* ...

### Task 2. Margins and margin width

The quantity $y_i f(x_i)$ is the (functional) **margin** of example $i$. SVM wants every point correct **with a safety margin**: $y_i f(x_i) \ge 1$.

| Line | Equation |
|---|---|
| decision boundary | $f(x) = 0$ |
| positive margin | $f(x) = +1$ |
| negative margin | $f(x) = -1$ |

The corridor between the two margin lines has width $\dfrac{2}{\|w\|}$. So **small weights ⇔ wide margin**.

In [ ]:
def margins(X, y, w, b):
    # TODO: the margin y_i * f(x_i) of every example, no loop
    raise NotImplementedError


def margin_width(w):
    # TODO: 2 / ||w||   (hint: np.linalg.norm)
    raise NotImplementedError


# Tiny example: 4 points, 2 classes
Xt = np.array([[0, 0], [-1, 0], [2, 2], [3, 2]], dtype=float)
yt = np.array([-1, -1, 1, 1])

for w_t, b_t in [(np.array([0.5, 0.5]), -1.0), (np.array([1.0, 1.0]), -2.0)]:
    print("w =", w_t, "b =", b_t, "-> margins", margins(Xt, yt, w_t, b_t),
          "width", round(margin_width(w_t), 3))

**Check yourself:** `[1. 1.5 1. 1.5]` with width `2.828`, then `[2. 3. 2. 3.]` with width `1.414`.

> Both $(w, b)$ pairs describe the **same line** and both satisfy $y_i f(x_i) \ge 1$ for every point. Which one would SVM choose, and why? Which points touch the margin lines in the first case?

*Your answer:* ...

### Visualizing the margin

The function below draws a linear model given by $(w, b)$: regions, boundary (solid), margins (dashed), and optionally circles some points. To see what we are aiming for, we borrow the answer from scikit-learn for one cell.

In [ ]:
# ---- PROVIDED
from sklearn.svm import SVC


def plot_svm(w, b, X, y, title, support=None, ax=None):
    if ax is None:
        ax = plt.gca()

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                           np.linspace(x2_min, x2_max, 300))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    F = (grid @ w + b).reshape(xx1.shape)

    ax.contourf(xx1, xx2, np.sign(F), alpha=0.2, cmap="coolwarm")
    ax.contour(xx1, xx2, F, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])

    neg, pos = (y <= 0), (y > 0)
    ax.scatter(X[neg, 0], X[neg, 1], c="tab:blue", edgecolor="k", label="class -1 / 0")
    ax.scatter(X[pos, 0], X[pos, 1], c="tab:red",  edgecolor="k", label="class +1 / 1")
    if support is not None:
        ax.scatter(X[support, 0], X[support, 1], s=200, facecolors="none",
                   edgecolors="k", linewidths=1.5, label="support vectors")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)


preview = SVC(kernel="linear", C=1.0).fit(X, y)
plot_svm(preview.coef_[0], preview.intercept_[0], X, y_svm,
         "Target picture: boundary, margins, support vectors", support=preview.support_)
plt.show()

Only a **few** circled points touch the dashed lines. These **support vectors** hold the boundary in place, like pins.

## Part B. The hinge loss

### Task 3. Hinge loss

> **How do we know if the prediction is good?** For SVM: correct **with a margin of at least 1**.

$$
L_i = \max\big(0,\; 1 - y_i f(x_i)\big)
$$

```text
margin y·f(x) ≥ 1   → loss = 0              correct and safe
0 < y·f(x) < 1      → small positive loss   correct, but inside the corridor
y·f(x) < 0          → large loss            wrong side of the boundary
```

In [ ]:
def hinge(margin):
    # TODO: element-wise max(0, 1 - margin)   (hint: np.maximum, not max)
    raise NotImplementedError


print(hinge(np.array([3.0, 1.0, 0.5, 0.0, -1.0, -3.0])))

# TODO: for margins in [-3, 3], plot on the same figure
#   - the hinge loss
#   - the logistic loss  np.log(1 + np.exp(-margins)) / np.log(2)   (dashed, for comparison)
# with vertical dotted lines at 0 and 1

**Check yourself:** `[0. 0. 0.5 1. 2. 4.]`.

> The logistic loss is never exactly 0, the hinge loss is. What does that mean for **which points influence the boundary** in each model?

*Your answer:* ...

## Part C. Build the model

### Task 4. The `LinearSVMScratch` class

The soft-margin SVM minimises

$$
J(w, b) = \underbrace{\frac12 \|w\|^2}_{\text{wide margin}} \;+\; C \,\underbrace{\frac1m \sum_{i=1}^{m} \max\big(0,\, 1 - y_i f(x_i)\big)}_{\text{few violations}}
$$

The hinge has a corner at margin = 1, so we use its slope on each side (a *subgradient*). Only the **violating** points (margin $< 1$) contribute:

$$
\nabla_w J = w - \frac{C}{m} \sum_{i:\, y_i f(x_i) < 1} y_i x_i,
\qquad
\frac{\partial J}{\partial b} = - \frac{C}{m} \sum_{i:\, y_i f(x_i) < 1} y_i
$$

$$
w \leftarrow w - \eta \nabla_w J, \qquad b \leftarrow b - \eta \frac{\partial J}{\partial b}
$$

Same structure as `LogisticRegressionScratch`: `__init__` stores hyperparameters, `fit` learns `self.w` and `self.b`, `predict` uses them. Only the loss and its gradient change.

Hint: if `violating` is a boolean mask, `y[violating] @ X[violating]` computes $\sum y_i x_i$ over the violating points.

In [ ]:
class LinearSVMScratch:

    def __init__(self, lr=0.01, epochs=1000, C=1.0):
        # TODO: store the hyperparameters only
        pass

    def fit(self, X, y):
        # y must contain -1 and +1
        m, n = X.shape

        # TODO: initialise self.w (zeros), self.b (0.0), self.loss_history (empty list)

        # TODO: for each epoch:
        #   1. scores f and margins y * f
        #   2. objective 0.5 * ||w||^2 + C * mean hinge; append it to self.loss_history
        #   3. boolean mask of violating points (margin < 1)
        #   4. dw and db (formulas above)
        #   5. update self.w and self.b
        raise NotImplementedError

        return self

    def predict(self, X):
        # TODO: +1 where the score is >= 0, -1 otherwise
        raise NotImplementedError

### Task 5. Train the scratch SVM

In [ ]:
model = LinearSVMScratch(lr=0.01, epochs=1000, C=1.0)

# TODO (1): train on (X, y_svm); compute pred on X and the training accuracy
pred = None

print("Learned w:", model.w)
print("Learned b:", round(model.b, 4))
print("Training accuracy:", np.mean(pred == y_svm))
print("Margin width:", round(margin_width(model.w), 3))

# TODO (2): plot model.loss_history

**Check yourself:** `w = [-0.1597 0.4387]`, `b ≈ 0.4256`, accuracy `1.0`, width `4.284`.

### Task 6. Which points matter?

For our model, the points that influence the boundary are those with $y_i f(x_i) \le 1$. Gradient descent never lands *exactly* on 1, so we use a small tolerance: $y_i f(x_i) \le 1.05$.

In [ ]:
# TODO: indices of the points with margin <= 1 + 0.05 (hint: margins(...) and np.where(...)[0])
support_scratch = None

print("Points on or inside the margin:", len(support_scratch))
plot_svm(model.w, model.b, X, y_svm, "Our LinearSVMScratch (C = 1)", support=support_scratch)
plt.show()

**Check yourself:** `37` points.

> The target picture had only 2 support vectors. Why does our model have so many points inside the corridor? (Hint: look at the objective. What does $C$ balance?)

*Your answer:* ...

## Part D. Understanding $C$

### Task 7. Three values of $C$

$C$ is the **price of a violation**.

- **Small $C$:** violations are cheap → mostly minimise $\|w\|$ → very wide margin, many points inside.
- **Large $C$:** violations are expensive → narrower margin, few violations.

Predict what you will see before running.

In [ ]:
C_values = [0.01, 1, 100]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, C in zip(axes, C_values):
    # TODO: train a LinearSVMScratch(lr=0.01, epochs=1000, C=C) on (X, y_svm),
    #       compute its margin width and the points with margin <= 1.05,
    #       then plot_svm(..., ax=ax) with both numbers in the title
    pass
plt.show()

**Check yourself:** widths `78.00`, `4.28`, `3.26`; points in the margin `100`, `37`, `2`.

> With $C = 0.01$ you cannot see the dashed lines at all. Where are they? On noisy data, which extreme risks **overfitting** and which risks **underfitting**?

*Your answer:* ...

## Part E. Compare with scikit-learn

### Task 8. `SVC` with a linear kernel

In [ ]:
# TODO (1): sk_model = SVC(kernel="linear", C=1.0), fit it on (X, y)   <- sklearn accepts {0, 1}
#           sk_pred on X; print coef_[0], intercept_[0], the accuracy, support_ and n_support_
sk_pred = None

# TODO (2): side by side, our model (support_scratch) and sklearn's (sk_model.support_)

**Check yourself:** sklearn `w = [-0.3374 0.4422]`, `b ≈ 0.878`, accuracy `1.0`, `support_ = [8 82]`, `n_support_ = [1 1]`.

### Task 9. Same letter $C$, different scale

Both models classify everything correctly, but the margins differ. Read the definitions carefully:

- our objective uses the **mean** hinge loss: $\;C \cdot \frac1m \sum_i \text{hinge}_i$
- sklearn's `SVC` uses the **sum**: $\;C \cdot \sum_i \text{hinge}_i$

> With $m = 100$ points, which value of **our** $C$ corresponds to sklearn's `C=1`? Verify it.

In [ ]:
C_ours = None   # TODO: your answer

# TODO: train LinearSVMScratch(lr=0.01, epochs=1000, C=C_ours) on (X, y_svm),
#       print its w and b next to sklearn's, and plot both side by side

**Check yourself:** `C_ours = 100` gives `w = [-0.3567 0.4989]`, `b ≈ 0.87`, very close to sklearn.

> Where does the remaining small difference come from? What is the general lesson about hyperparameters in libraries?

*Your answer:* ...

## Part F. Nonlinear data

### Task 10. When a line is not enough

In [ ]:
# ---- PROVIDED
from sklearn.datasets import make_circles

X_c, y_c = make_circles(
    n_samples=100,
    noise=0.08,
    factor=0.4,
    random_state=42
)

plt.scatter(X_c[y_c == 0, 0], X_c[y_c == 0, 1], c="tab:blue", edgecolor="k", label="class 0")
plt.scatter(X_c[y_c == 1, 0], X_c[y_c == 1, 1], c="tab:red",  edgecolor="k", label="class 1")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()


def plot_kernel_svm(model, X, y, title, ax=None):
    if ax is None:
        ax = plt.gca()
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                           np.linspace(x2_min, x2_max, 300))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    Z = model.predict(grid).reshape(xx1.shape)
    F = model.decision_function(grid).reshape(xx1.shape)

    ax.contourf(xx1, xx2, Z, alpha=0.2, cmap="coolwarm")
    ax.contour(xx1, xx2, F, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red",  edgecolor="k")
    ax.scatter(X[model.support_, 0], X[model.support_, 1], s=200,
               facecolors="none", edgecolors="k", linewidths=1.2)
    ax.set_aspect("equal")
    ax.set_title(title)

> **Can a straight line solve this problem?**

*Your answer:* ...

`SVC` can use **kernels**, which let the SVM draw curved boundaries. Think of a kernel as *"a different way of measuring similarity between points"*. The interface does not change: `fit`, `predict`.

In [ ]:
# TODO: complete the dictionary with three classifiers:
#   "linear"         -> SVC(kernel="linear")
#   "poly, degree 2" -> SVC(kernel="poly", degree=2, C=1)
#   "rbf"            -> SVC(kernel="rbf", C=1)
kernels = {}

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, (name, clf) in zip(axes, kernels.items()):
    # TODO: fit clf on (X_c, y_c), compute its training accuracy, then plot_kernel_svm(..., ax=ax)
    pass
plt.show()

**Check yourself:** training accuracy `0.56` (linear), `1.00` (poly, degree 2), `1.00` (rbf).

> Why is a degree-2 polynomial a natural choice for this particular dataset?

*Your answer:* ...

## Part G. Small experiments

Write your prediction **before** running each cell.

### Task 11. One annoying point

We add a single point labelled $-1$ in the gap, very close to the red cloud, and train with a small and a large $C$. (The learning rate is lowered because large $C$ makes the steps large.)

In [ ]:
# ---- PROVIDED
outlier = np.array([[2.5, 1.4]])
X_out = np.vstack([X, outlier])
y_out = np.append(y_svm, -1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, C in zip(axes, [1, 1000]):
    m_out = LinearSVMScratch(lr=0.001, epochs=5000, C=C).fit(X_out, y_out)
    acc = np.mean(m_out.predict(X_out) == y_out)
    plot_svm(m_out.w, m_out.b, X_out, y_out, f"C = {C}, accuracy = {acc:.2f}", ax=ax)
    ax.scatter(*outlier[0], marker="*", s=400, c="yellow", edgecolor="k")
plt.show()

*Your prediction and observation:* ...

### Task 12. The `gamma` of the RBF kernel

`gamma` controls how far the influence of each training point reaches.

In [ ]:
# TODO: for gamma in [0.1, 1, 50], train SVC(kernel="rbf", C=1, gamma=gamma) on (X_c, y_c)
#       and plot it with plot_kernel_svm in a 1x3 figure, training accuracy in each title

**Check yourself:** training accuracy `0.98`, `1.00`, `1.00`.

*Your prediction and observation:* ...

## Final comparison

| | Logistic Regression | SVM |
|---|---|---|
| Score | $z = w^Tx + b$ | $f(x) = w^Tx + b$ |
| Output | $p = \sigma(z)$, a probability | sign of $f(x)$, a class |
| Labels | $\{0, 1\}$ | $\{-1, +1\}$ |
| Loss | binary cross-entropy | hinge: $\max(0, 1 - y f(x))$ |
| Who influences the boundary | every point, a little | only the support vectors |
| Main idea | **model the probability of class 1** | **find a boundary with a large margin** |

```python
model.fit(X, y)
model.predict(X)
```

> **Different algorithms can have the same interface.**